# CHEM 269 — Tier-2 CREST+ALPB: 5 Reference Compounds

Runs CREST dual-dielectric conformer sampling on:
- **CycloA** (Cyclosporin A) — gold-standard chameleon, literature ΔPSA ~75 Å²
- **1NMe3** — N-methylated analog, permeable
- **HexPep** — parent hexapeptide, impermeable
- **DP172** — highly permeable pharmaceutical
- **c*[PSLYF]** — impermeable control

All 5 compounds run **in parallel** (~1 hour total vs ~3 hours sequential).

---

### Setup
1. `Runtime → Change runtime type` → any CPU or GPU runtime (CPU-only is fine)
2. Run cells top to bottom
3. After Cell 1 restarts the runtime, skip back to **Cell 2**
4. Optionally mount Drive (Cell 3) to auto-save results — or just download at the end

### Output
- `tier2_reference_results.csv` — CREST ΔPSA, ΔHB, shape descriptors for all 5 compounds
- `tier2_reference_summary.txt` — comparison vs literature expected values

In [ ]:
# ── CELL 1: Install condacolab (runtime will restart automatically) ───────────
# Only runs once. After the automatic restart, skip to Cell 2.
try:
    import condacolab
    print('condacolab already installed — skip to Cell 2')
except ImportError:
    !pip install -q condacolab
    import condacolab
    condacolab.install()  # triggers automatic runtime restart

In [ ]:
# ── CELL 2: Install CREST + xtb + RDKit (~5-8 min, once per session) ─────────
import subprocess, sys

print('Installing crest + xtb + rdkit via mamba...')
subprocess.run(
    ['mamba', 'install', '-c', 'conda-forge', 'crest', 'xtb', 'rdkit', '-y', '-q'],
    check=True
)

r = subprocess.run(['crest', '--version'], capture_output=True, text=True)
print('CREST:', r.stdout.strip() or r.stderr.strip())
r = subprocess.run(['xtb', '--version'], capture_output=True, text=True)
print('xtb:  ', r.stdout.strip()[:80])
import rdkit
print('RDKit:', rdkit.__version__)
import multiprocessing, psutil
print(f'CPUs : {multiprocessing.cpu_count()}')
print(f'RAM  : {psutil.virtual_memory().total/1e9:.0f} GB')

In [ ]:
# ── CELL 3: Mount Google Drive (optional but recommended) ─────────────────────
# Results auto-save to Drive so you won't lose them if the session ends.
# Skip this cell if you prefer to just download at the end.

USE_DRIVE = True   # set False to skip Drive and save locally only

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS_DIR = '/content/drive/MyDrive/chem269_tier2/results/'
    import os; os.makedirs(RESULTS_DIR, exist_ok=True)
    print(f'Drive mounted. Results will save to: {RESULTS_DIR}')
else:
    RESULTS_DIR = '/content/tier2_results/'
    import os; os.makedirs(RESULTS_DIR, exist_ok=True)
    print(f'Drive skipped. Results saved locally at: {RESULTS_DIR}')

In [ ]:
# ── CELL 4: Configuration ─────────────────────────────────────────────────────

# Threads per compound for CREST.
# All 5 compounds run in parallel, so total CPUs used = N_THREADS_PER_COMPOUND * 5
# Default: 2 threads each = 10 CPUs total (safe for any runtime)
N_THREADS_PER_COMPOUND = 2

# Working directory for CREST temp files (local = fast I/O, discarded on session end)
WORK_ROOT = '/tmp/crest_runs'

# Output file paths
RESULTS_CSV     = RESULTS_DIR + 'tier2_reference_results.csv'
RESULTS_SUMMARY = RESULTS_DIR + 'tier2_reference_summary.txt'

import os
os.makedirs(WORK_ROOT, exist_ok=True)
print(f'N_THREADS_PER_COMPOUND : {N_THREADS_PER_COMPOUND}')
print(f'Total CPUs target      : {N_THREADS_PER_COMPOUND * 5}')
print(f'Results CSV            : {RESULTS_CSV}')

In [ ]:
# ── CELL 5: Processing functions (self-contained, no colab_utils needed) ──────

import os, subprocess, logging, threading
import numpy as np
from pathlib import Path
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Descriptors3D, rdFreeSASA
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Geometry import rdGeometry

RDLogger.DisableLog('rdApp.*')
logging.basicConfig(level=logging.INFO, format='%(levelname)s [%(name)s]: %(message)s')
_log = logging.getLogger('tier2')

_BONDI = {'H':1.20,'C':1.70,'N':1.55,'O':1.52,'S':1.80,'P':1.80,'F':1.47,'Cl':1.75,'Br':1.85,'I':1.98}
_POLAR = {'N','O','S','P'}
HB_DONOR    = Chem.MolFromSmarts('[N,O;!H0]')
HB_ACCEPTOR = Chem.MolFromSmarts('[N,O]')


def smiles_to_xyz(smiles, mol_id, work_dir):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None, None
    try:
        mol = rdMolStandardize.FragmentParent(mol)
        mol = rdMolStandardize.Uncharger(canonicalOrder=True).uncharge(mol)
        Chem.SanitizeMol(mol)
    except Exception:
        pass
    mol_h = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    params.useMacrocycleTorsions = True
    params.useSmallRingTorsions  = True
    params.maxIterations = 2000
    if AllChem.EmbedMolecule(mol_h, params) != 0:
        return None, None
    AllChem.MMFFOptimizeMolecule(mol_h, mmffVariant='MMFF94s', maxIters=2000)
    conf = mol_h.GetConformer()
    lines = [str(mol_h.GetNumAtoms()), f'ID={mol_id}']
    for atom in mol_h.GetAtoms():
        p = conf.GetAtomPosition(atom.GetIdx())
        lines.append(f'{atom.GetSymbol()}  {p.x:.6f}  {p.y:.6f}  {p.z:.6f}')
    xyz_path = Path(work_dir) / f'{mol_id}_start.xyz'
    xyz_path.write_text('\n'.join(lines))
    return xyz_path, mol_h


def run_crest(xyz_path, solvent, n_threads, run_dir):
    """Run CREST --alpb <solvent>. Returns output dir or None."""
    solvent_dir = Path(run_dir) / solvent
    solvent_dir.mkdir(parents=True, exist_ok=True)
    cmd = ['crest', str(xyz_path), '--alpb', solvent, '--T', str(n_threads), '--quick']
    log_path = solvent_dir / 'crest.log'
    try:
        with open(log_path, 'w') as lf:
            r = subprocess.run(cmd, cwd=str(solvent_dir), stdout=lf,
                               stderr=subprocess.STDOUT, timeout=3600)
        if r.returncode != 0:
            _log.warning('CREST non-zero return: %s solvent=%s', xyz_path.stem, solvent)
            return None
        ens = solvent_dir / 'crest_conformers.xyz'
        return solvent_dir if ens.exists() else None
    except subprocess.TimeoutExpired:
        _log.warning('CREST timeout: %s solvent=%s', xyz_path.stem, solvent)
        return None
    except FileNotFoundError:
        _log.error('crest binary not found — run Cell 2 first')
        return None


def parse_crest_best(crest_dir):
    """Return lowest-energy conformer XYZ block (first in ensemble file)."""
    path = Path(crest_dir) / 'crest_conformers.xyz'
    if not path.exists():
        return None
    lines = path.read_text().strip().split('\n')
    i = 0
    while i < len(lines):
        try:
            n = int(lines[i].strip())
        except ValueError:
            i += 1
            continue
        block = lines[i: i + n + 2]
        if len(block) == n + 2:
            return '\n'.join(block)
        i += n + 2
    return None


def xyz_to_mol(xyz_block, template_mol):
    """Apply CREST coordinates onto RDKit template mol."""
    lines = xyz_block.strip().split('\n')
    try:
        n = int(lines[0].strip())
    except ValueError:
        return None
    if n != template_mol.GetNumAtoms():
        return None
    coords, elems = [], []
    for line in lines[2: 2 + n]:
        parts = line.split()
        if len(parts) < 4:
            return None
        elems.append(parts[0])
        coords.append((float(parts[1]), float(parts[2]), float(parts[3])))
    for i, (atom, elem) in enumerate(zip(template_mol.GetAtoms(), elems)):
        if atom.GetSymbol() != elem:
            return None
    rw = Chem.RWMol(template_mol)
    rw.RemoveAllConformers()
    conf = Chem.Conformer(n)
    for i, (x, y, z) in enumerate(coords):
        conf.SetAtomPosition(i, rdGeometry.Point3D(x, y, z))
    rw.AddConformer(conf, assignId=True)
    return rw.GetMol()


def polar_sasa(mol, conf_id=0):
    try:
        radii = []
        for atom in mol.GetAtoms():
            sym = atom.GetSymbol()
            radii.append(_BONDI.get(sym, 1.50))
            if sym in _POLAR:
                atom.SetIntProp('SASAClass', 0)
                atom.SetProp('SASAClassName', 'Polar')
            else:
                atom.SetIntProp('SASAClass', 1)
                atom.SetProp('SASAClassName', 'APolar')
        query = rdFreeSASA.MakeFreeSasaPolarAtomQuery()
        return round(rdFreeSASA.CalcSASA(mol, radii, confIdx=conf_id, query=query), 4)
    except Exception:
        return np.nan


def intramolecular_hbonds(mol, conf_id=0):
    try:
        pos       = mol.GetConformer(conf_id).GetPositions()
        donors    = [i for m in mol.GetSubstructMatches(HB_DONOR)    for i in m]
        acceptors = [i for m in mol.GetSubstructMatches(HB_ACCEPTOR) for i in m]
        count = 0
        for d in donors:
            for h in mol.GetAtomWithIdx(d).GetNeighbors():
                if h.GetAtomicNum() != 1:
                    continue
                h_pos, d_pos = pos[h.GetIdx()], pos[d]
                for a in acceptors:
                    if a == d:
                        continue
                    try:
                        if len(Chem.GetShortestPath(mol, d, a)) < 6:
                            continue
                    except Exception:
                        continue
                    if np.linalg.norm(h_pos - pos[a]) > 3.0:
                        continue
                    vhd = d_pos - h_pos
                    vha = pos[a] - h_pos
                    cos = np.dot(vhd, vha) / (np.linalg.norm(vhd) * np.linalg.norm(vha) + 1e-9)
                    if np.degrees(np.arccos(np.clip(cos, -1, 1))) >= 120.0:
                        count += 1
        return count
    except Exception:
        return np.nan


def shape_descriptors(mol, conf_id=0):
    try:
        return {
            'Rg':          Descriptors3D.RadiusOfGyration(mol, confId=conf_id),
            'NPR1':        Descriptors3D.NPR1(mol, confId=conf_id),
            'NPR2':        Descriptors3D.NPR2(mol, confId=conf_id),
            'Asphericity': Descriptors3D.Asphericity(mol, confId=conf_id),
        }
    except Exception:
        return {k: np.nan for k in ['Rg','NPR1','NPR2','Asphericity']}


def process_compound(compound):
    """
    Full CREST+ALPB pipeline for one compound dict.
    Runs water and chcl3 sequentially; called in parallel across compounds.
    Returns result dict.
    """
    import time
    mol_id = compound['id']
    smiles = compound['smiles']
    t0     = time.time()

    print(f'  [{mol_id}] Starting...', flush=True)

    work_dir = Path(WORK_ROOT) / str(mol_id)
    work_dir.mkdir(parents=True, exist_ok=True)

    base = {'id': mol_id, 'name': compound['name'], 'pampa': compound['pampa'],
            'permeable': compound['permeable'], 'error': None}

    xyz_path, template_mol = smiles_to_xyz(smiles, mol_id, work_dir)
    if xyz_path is None:
        print(f'  [{mol_id}] FAILED: embed_failed', flush=True)
        return {**base, 'error': 'embed_failed'}

    results_by_solvent = {}
    for solvent in ('water', 'chcl3'):
        print(f'  [{mol_id}] Running CREST --alpb {solvent}...', flush=True)
        crest_dir = run_crest(xyz_path, solvent, N_THREADS_PER_COMPOUND, work_dir)
        if crest_dir is None:
            print(f'  [{mol_id}] FAILED: crest_failed_{solvent}', flush=True)
            return {**base, 'error': f'crest_failed_{solvent}'}
        xyz_block = parse_crest_best(crest_dir)
        if xyz_block is None:
            return {**base, 'error': f'parse_failed_{solvent}'}
        mol_out = xyz_to_mol(xyz_block, template_mol)
        if mol_out is None:
            return {**base, 'error': f'coord_failed_{solvent}'}
        results_by_solvent[solvent] = {
            'psa': polar_sasa(mol_out),
            'hb':  intramolecular_hbonds(mol_out),
            **shape_descriptors(mol_out)
        }

    aq  = results_by_solvent['water']
    mem = results_by_solvent['chcl3']

    delta_psa = float(aq['psa'] - mem['psa']) if not (np.isnan(aq['psa']) or np.isnan(mem['psa'])) else np.nan
    elapsed   = round(time.time() - t0, 1)
    print(f'  [{mol_id}] Done in {elapsed:.0f}s  aq_psa={aq["psa"]:.1f}  mem_psa={mem["psa"]:.1f}  ΔPSA={delta_psa:.1f}', flush=True)

    return {
        **base,
        'aq_psa3d':       aq['psa'],
        'aq_hb_count':    aq['hb'],
        'aq_Rg':          aq['Rg'],
        'aq_NPR1':        aq['NPR1'],
        'aq_NPR2':        aq['NPR2'],
        'aq_Asphericity': aq['Asphericity'],
        'mem_psa3d':      mem['psa'],
        'mem_hb_count':   mem['hb'],
        'mem_Rg':         mem['Rg'],
        'mem_NPR1':       mem['NPR1'],
        'mem_NPR2':       mem['NPR2'],
        'mem_Asphericity':mem['Asphericity'],
        'delta_psa3d':    delta_psa,
        'delta_hb':       float(mem['hb'] - aq['hb']) if not (np.isnan(aq['hb']) or np.isnan(mem['hb'])) else np.nan,
        'delta_Rg':       float(aq['Rg'] - mem['Rg']) if not (np.isnan(aq['Rg']) or np.isnan(mem['Rg'])) else np.nan,
        'wall_s':         elapsed,
        'solvent_aq':     'water (eps=80)',
        'solvent_mem':    'chcl3 (eps=4.8)',
    }


print('Processing functions loaded.')

In [ ]:
# ── CELL 6: Reference compound definitions ────────────────────────────────────
# Hardcoded — no CSV upload needed.
# SMILES from data/reference_set.csv (canonical, RDKit-standardized).

REFERENCE_COMPOUNDS = [
    {
        'id':         'HexPep',
        'name':       'Hexapeptide (c[dL-dL-L-dL-P-Y])',
        'smiles':     'CC(C)C[C@@H]1NC(=O)[C@@H](CC(C)C)NC(=O)[C@@H](CC(C)C)NC(=O)[C@H](Cc2ccc(O)cc2)NC(=O)[C@@H]2CCCN2C(=O)[C@@H](CC(C)C)NC1=O',
        'pampa':      -6.2,
        'permeable':  False,
        'lit_delta_psa': None,
        'lit_source': 'Rezai & Lokey, JACS 2006',
    },
    {
        'id':         '1NMe3',
        'name':       'N-Me Hexapeptide (1NMe3)',
        'smiles':     'CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)N(C)C(=O)[C@H]2CCCN2C(=O)[C@H](CC(C)C)NC(=O)[C@H](CC(C)C)N(C)C(=O)[C@@H](CC(C)C)N(C)C1=O',
        'pampa':      -5.52,
        'permeable':  True,
        'lit_delta_psa': None,
        'lit_source': 'White & Lokey, Nat Chem Biol 2011',
    },
    {
        'id':         'CsA',
        'name':       'Cyclosporin A',
        'smiles':     'C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC)C(=O)N(C)CC(=O)N(C)[C@@H](CC(C)C)C(=O)N[C@@H](C(C)C)C(=O)N(C)[C@@H](CC(C)C)C(=O)N[C@@H](C)C(=O)N[C@H](C)C(=O)N(C)[C@@H](CC(C)C)C(=O)N(C)[C@@H](CC(C)C)C(=O)N(C)[C@@H](C(C)C)C(=O)N1C',
        'pampa':      -6.6,
        'permeable':  True,
        'lit_delta_psa': 75.0,   # Witek et al. JCTC 2016 (MD+explicit solvent)
        'lit_source': 'Witek et al., JCTC 2016',
    },
    {
        'id':         'DP172',
        'name':       'DP-172',
        'smiles':     'CC[C@H](C)[C@@H]1NC(=O)[C@H]([C@@H](C)O)NC(=O)[C@H](C)N(C)C(=O)[C@H](CC(C)C)N(C)C(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C@H](C)N(C)C(=O)[C@H](CC(C)C)N(C)C(=O)[C@@H](C(C)C)NC(=O)C[C@@H](C(=O)N2CCCCC2)NC1=O',
        'pampa':      -4.15,
        'permeable':  True,
        'lit_delta_psa': None,
        'lit_source': 'CHUGAI 2013 pharmaceutical screen',
    },
    {
        'id':         'PSLYF',
        'name':       'c*[PSLYF]',
        'smiles':     'CC(C)C[C@@H]1NC(=O)[C@H](CO)NC(=O)[C@@H]2CCCN2[C@H](C(=O)NC(C)(C)C)[C@H](C)NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H](Cc2ccc(O)cc2)NC1=O',
        'pampa':      -9.1,
        'permeable':  False,
        'lit_delta_psa': None,
        'lit_source': 'Hickey, J Med Chem 2016',
    },
]

print(f'Loaded {len(REFERENCE_COMPOUNDS)} reference compounds:')
for c in REFERENCE_COMPOUNDS:
    lit = f"  (lit ΔPSA ~{c['lit_delta_psa']:.0f} Å²)" if c['lit_delta_psa'] else ''
    perm = 'permeable' if c['permeable'] else 'impermeable'
    print(f"  {c['id']:<10}  PAMPA={c['pampa']:.2f}  {perm}{lit}")

In [ ]:
# ── CELL 7: Run all 5 compounds in parallel ───────────────────────────────────
# All 5 compounds start simultaneously.
# Each uses N_THREADS_PER_COMPOUND CREST threads.
# Results are saved to Drive (or locally) as each compound finishes.

import time
import pandas as pd
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

results     = []
save_lock   = threading.Lock()
t_start     = time.time()

def save_result(result):
    """Thread-safe append of one result row to the CSV."""
    with save_lock:
        row = pd.DataFrame([result])
        write_header = not Path(RESULTS_CSV).exists()
        row.to_csv(RESULTS_CSV, mode='a', header=write_header, index=False)


print(f'Launching {len(REFERENCE_COMPOUNDS)} compounds in parallel '
      f'({N_THREADS_PER_COMPOUND} CREST threads each)...')
print('-' * 60)

with ThreadPoolExecutor(max_workers=len(REFERENCE_COMPOUNDS)) as executor:
    future_to_compound = {
        executor.submit(process_compound, c): c['id']
        for c in REFERENCE_COMPOUNDS
    }

    for future in as_completed(future_to_compound):
        mol_id = future_to_compound[future]
        try:
            result = future.result()
            results.append(result)
            save_result(result)   # save to Drive immediately
            status = result.get('error') or 'OK'
            dpsa   = result.get('delta_psa3d', float('nan'))
            wall   = result.get('wall_s', 0)
            print(f'FINISHED: {mol_id:<10}  ΔPSA={dpsa:>7.1f} Å²  '
                  f't={wall:.0f}s  status={status}')
        except Exception as e:
            print(f'ERROR: {mol_id} raised exception: {e}')
            results.append({'id': mol_id, 'error': str(e)})

total_elapsed = time.time() - t_start
print('-' * 60)
print(f'All done in {total_elapsed/60:.1f} min')
print(f'Results saved to: {RESULTS_CSV}')

In [ ]:
# ── CELL 8: Results summary and literature comparison ─────────────────────────

import pandas as pd
import numpy as np
from pathlib import Path

res = pd.read_csv(RESULTS_CSV)
ok  = res[res['error'].isna()].copy()

print(f'Processed: {len(res)}  Successful: {len(ok)}  Failed: {res["error"].notna().sum()}')
if res['error'].notna().any():
    print('Failures:', res[res['error'].notna()][['id','error']].to_string(index=False))

# Build comparison table
lit_map = {c['id']: c.get('lit_delta_psa') for c in REFERENCE_COMPOUNDS}
perm_map = {c['id']: c['permeable'] for c in REFERENCE_COMPOUNDS}

rows = []
for _, r in ok.iterrows():
    lit = lit_map.get(r['id'])
    rows.append({
        'ID':          r['id'],
        'PAMPA':       r['pampa'],
        'Permeable':   '✓' if perm_map.get(r['id']) else '✗',
        'aq_PSA (Å²)': f"{r['aq_psa3d']:.1f}",
        'mem_PSA (Å²)':f"{r['mem_psa3d']:.1f}",
        'ΔPSA (Å²)':   f"{r['delta_psa3d']:.1f}",
        'ΔHB':         f"{r['delta_hb']:.0f}",
        'Lit ΔPSA':    f"~{lit:.0f}" if lit else '—',
    })

table = pd.DataFrame(rows).sort_values('PAMPA', ascending=False)
print('\n' + '='*70)
print('CREST+ALPB Results vs Literature')
print('='*70)
print(table.to_string(index=False))
print('='*70)

# Key finding: permeable vs impermeable ΔPSA
if len(ok) >= 2:
    ok['permeable_bool'] = ok['id'].map(perm_map)
    grp = ok.groupby('permeable_bool')['delta_psa3d'].mean()
    print(f'\nMean ΔPSA — permeable: {grp.get(True, float("nan")):.1f} Å²  '
          f'impermeable: {grp.get(False, float("nan")):.1f} Å²')
    if True in grp and False in grp:
        diff = grp[True] - grp[False]
        direction = 'higher' if diff > 0 else 'lower'
        print(f'→ Permeable compounds show {abs(diff):.1f} Å² {direction} ΔPSA '
              f'({"consistent" if diff > 0 else "inconsistent"} with chameleonic hypothesis)')

# Save summary text
summary = table.to_string(index=False)
Path(RESULTS_SUMMARY).write_text(summary)
print(f'\nSummary saved to: {RESULTS_SUMMARY}')

In [ ]:
# ── CELL 9: Download results to local machine ─────────────────────────────────
# Files are already on Drive (if Cell 3 was run).
# This also downloads them directly to your computer.

from google.colab import files
from pathlib import Path

for p in [RESULTS_CSV, RESULTS_SUMMARY]:
    if Path(p).exists():
        print(f'Downloading {Path(p).name} ({Path(p).stat().st_size/1e3:.1f} KB)...')
        files.download(p)
    else:
        print(f'Not found: {p}')